# LPGNet — released implementation and factorial leakage study

This notebook follows `devalab/parkinsonsfromgait` (`Original.ipynb` and
`generateLPresidual.m`) and separates **participant overlap**, **preprocessing leakage**,
and **test-set model selection**.

| Condition | LP fitting | Early stopping / LR scheduling in both stages | Interpretation |
|---|---|---|---|
| `global__test` | All controls, once outside CV | Outer-test participants | Closest released protocol |
| `local__test` | Outer-training controls only | Outer-test participants | Isolates global-LP effect under test validation |
| `global__inner` | All controls, once outside CV | Held-out outer-training participants | Isolates test-validation effect under global LP |
| `local__inner` | Outer-training controls only | Held-out outer-training participants | Corrected outer-test protocol |

All outer folds are participant-disjoint. The global-LP conditions nevertheless leak
information from held-out controls into their feature extractor. Test validation leaks
outer-test labels into model selection. The two effects are not assumed to act only on
specificity: that is an empirical question.

The second stage is the released **Keras** stage, not logistic regression: frozen CNN,
new dropout and sigmoid dense layer, wrap-padded recordings, Adam, label-smoothed BCE,
and the same validation participants used in stage 1. Convolutions use the released
`valid` padding and official window endpoint convention. GPU nondeterminism and a
modern-Keras regularizer compatibility fallback can still prevent bit-identical scores.


In [ ]:
# ============================================================
# Reproducibility and experiment settings
# ============================================================
import os, time, random, json, urllib.request, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import lfilter
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (roc_auc_score, accuracy_score, balanced_accuracy_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             log_loss, roc_curve)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 1                 # official LPGNet notebook seed
FOLD_SEED = 0            # official StratifiedKFold seed
N_FOLDS = 10
N_INNER = 3
VAL_FRAC = 0.15
LP_ORDER = 11
WIN, STRIDE = 100, 50
BATCH_WIN, BATCH_REC = 128, 64
LR_WIN, LR_REC = 5e-4, 1e-3
LABEL_SMOOTH = 0.1
L2_REG = 1e-3
MAX_EPOCHS_WIN, MAX_EPOCHS_REC = 1000, 200
PATIENCE, RLR_PATIENCE, MIN_LR = 25, 14, 1e-7
N_BOOT = 1000
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# ============================================================
# Download PhysioNet data and attach verified clinical metadata
# ============================================================
URL = 'https://physionet.org/static/published-projects/gaitpdb/gait-in-parkinsons-disease-1.0.0.zip'
ZIP = 'gaitpdb.zip'; DATA_DIR = 'gaitpdb_data'
ROOT = os.path.join(DATA_DIR, 'gait-in-parkinsons-disease-1.0.0')
if not os.path.isdir(ROOT):
    if not os.path.exists(ZIP):
        print('Downloading PhysioNet gaitpdb (~300 MB, one-off)...')
        urllib.request.urlretrieve(URL, ZIP)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA_DIR)

records = []
for fn in sorted(os.listdir(ROOT)):
    if not fn.endswith('.txt') or fn[2:4] not in ('Co', 'Pt'):
        continue
    base = fn[:-4]
    arr = np.loadtxt(os.path.join(ROOT, fn), delimiter='\t')
    records.append(dict(file=fn, study=base[:2], subject=base.split('_')[0],
                        label=int(base[2:4] == 'Pt'),
                        signal=arr[:, 1:19].astype(np.float32)))
subjects = sorted({r['subject'] for r in records})
subj_label = {s: next(r['label'] for r in records if r['subject'] == s) for s in subjects}
by_file = {r['file']: i for i, r in enumerate(records)}
by_subject = {s: [i for i, r in enumerate(records) if r['subject'] == s] for s in subjects}
print(f"{len(records)} recordings | {len(subjects)} participants with sensor files | "
      f"{sum(subj_label.values())} PD / {len(subjects)-sum(subj_label.values())} controls")

# demographics.txt is distributed with gaitpdb. Missing values remain missing.
demo_path = os.path.join(ROOT, 'demographics.txt')
demo = pd.read_csv(demo_path, sep='\t', engine='python', usecols=range(20))
demo = demo.loc[:, ~demo.columns.str.startswith('Unnamed')].copy()
demo = demo.dropna(subset=['ID']).copy()
demo['ID'] = demo['ID'].astype(str)
demo = demo.drop_duplicates('ID').set_index('ID')
clinical = {}
for s in subjects:
    row = demo.loc[s] if s in demo.index else None
    hy = pd.to_numeric(row.get('HoehnYahr'), errors='coerce') if row is not None else np.nan
    # Controls are not assigned HY=0 by inference. Only recorded PD values are used.
    verified_hy = float(hy) if subj_label[s] == 1 and pd.notna(hy) else np.nan
    clinical[s] = dict(
        study=s[:2], diagnosis='PD' if subj_label[s] else 'Control',
        hy=verified_hy, hy_label=(f'{verified_hy:g}' if pd.notna(verified_hy)
                                  else ('Unknown' if subj_label[s] else 'Not applicable')),
        hy_status=('verified' if pd.notna(verified_hy) else
                   ('missing' if subj_label[s] else 'not_applicable')),
        age=pd.to_numeric(row.get('Age'), errors='coerce') if row is not None else np.nan,
        gender=pd.to_numeric(row.get('Gender'), errors='coerce') if row is not None else np.nan)
missing_metadata = [s for s in subjects if s not in demo.index]
print('Participants without a demographics join:', missing_metadata)


In [ ]:
# ============================================================
# Official preprocessing and LP residuals
# ============================================================
def preprocess(x):
    # Causal two-tap moving average, then decimation 100 -> 50 Hz.
    return lfilter([0.5, 0.5], 1, x, axis=0)[::2]

def fit_lp(control_sigs, order=LP_ORDER):
    """One predictor per channel; MATLAB-compatible pooled-control fitting."""
    C, conc_all = np.zeros((18, order)), []
    for c in range(18):
        # generateLPresidual.m starts with zero padding and inserts it between walks.
        pieces = [np.zeros(order, np.float32)]
        pieces += [np.concatenate([s[:, c], np.zeros(order, np.float32)])
                   for s in control_sigs]
        conc = np.concatenate(pieces)
        conc_all.append(conc)
        n = len(conc)
        nfft = 1 << (2*n - 1).bit_length()
        f = np.fft.rfft(conc, nfft)
        r = np.fft.irfft(f*np.conj(f), nfft)[:order+1] / n
        R = np.array([[r[abs(i-j)] for j in range(order)] for i in range(order)])
        C[c] = np.linalg.solve(R, r[1:order+1])
    # MATLAB std uses N-1 normalization. No test recordings enter this calculation.
    stds = np.std(np.stack(conc_all, axis=1), axis=0, ddof=1) + 1e-8
    return C, stds

def apply_lp(x, C, stds):
    e = np.stack([lfilter(np.r_[1.0, -C[c]], 1, x[:, c]) for c in range(18)], axis=1)
    return (e / stds).astype(np.float32)

def lp_features(indices, C, stds):
    return {i: apply_lp(records[i]['proc'], C, stds) for i in indices}

for r in records:
    r['proc'] = preprocess(r['signal'])


In [ ]:
# ============================================================
# Fixed participant-disjoint outer folds and assertions
# ============================================================
def make_folds():
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=FOLD_SEED)
    arr = np.array(subjects); y = np.array([subj_label[s] for s in subjects])
    folds = []
    for tr_s, te_s in skf.split(arr, y):
        tr_subj, te_subj = set(arr[tr_s]), set(arr[te_s])
        assert not tr_subj & te_subj
        tr = [i for i, r in enumerate(records) if r['subject'] in tr_subj]
        te = [i for i, r in enumerate(records) if r['subject'] in te_subj]
        assert not ({records[i]['subject'] for i in tr} & {records[i]['subject'] for i in te})
        folds.append(dict(train=tr, test=te))
    return folds

def split_val(indices, frac=VAL_FRAC, seed=SEED):
    """Split participants, never recordings; output participant sets are disjoint."""
    ss = sorted({records[i]['subject'] for i in indices})
    yy = [subj_label[s] for s in ss]
    if len(set(yy)) > 1:
        splitter = StratifiedShuffleSplit(1, test_size=frac, random_state=seed)
        tr_s, va_s = next(splitter.split(np.zeros(len(ss)), yy))
    else:
        perm = np.random.default_rng(seed).permutation(len(ss))
        n_va = max(1, int(round(frac*len(ss))))
        va_s, tr_s = perm[:n_va], perm[n_va:]
    tr_subj, va_subj = {ss[i] for i in tr_s}, {ss[i] for i in va_s}
    assert not tr_subj & va_subj
    return ([i for i in indices if records[i]['subject'] in tr_subj],
            [i for i in indices if records[i]['subject'] in va_subj])

def labels(indices):
    return np.array([records[i]['label'] for i in indices], dtype=np.float32)

FOLDS = make_folds()
for k, f in enumerate(FOLDS):
    tr_n = len({records[i]['subject'] for i in f['train']})
    te_n = len({records[i]['subject'] for i in f['test']})
    print(f"Fold {k}: {tr_n} train participants / {te_n} test participants | "
          f"{len(f['train'])} train / {len(f['test'])} test recordings")


In [ ]:
# ============================================================
# Released LPGNet architecture and official two-stage training
# ============================================================
def official_sepconv(filters, kernel_size, reg):
    """Use the released call; provide an equivalent modern-Keras fallback."""
    try:
        return layers.SeparableConv1D(filters, kernel_size, activation='linear',
                                      padding='valid', kernel_regularizer=reg)
    except TypeError:
        # Newer Keras exposes separate depthwise/pointwise regularizer arguments.
        return layers.SeparableConv1D(filters, kernel_size, activation='linear',
                                      padding='valid', depthwise_regularizer=reg,
                                      pointwise_regularizer=reg)

def build_lpgnet(T=None, n_ch=18, channels=32, kernels=(7,5,3), spat_drop=0.2,
                 drop=0.25, l2=L2_REG, noise=0.01):
    reg = keras.regularizers.l2(l2)
    if isinstance(channels, int):
        channels = [channels]*(len(kernels)-1) + [2*channels]
    inp = layers.Input((None, n_ch))
    x = layers.SpatialDropout1D(spat_drop)(inp)
    x = layers.GaussianNoise(noise)(x)
    for i, (k, c) in enumerate(zip(kernels, channels)):
        x = official_sepconv(c, k, reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('elu', name='embedding' if i == len(kernels)-1 else None)(x)
        if i < len(kernels)-1:
            x = layers.MaxPool1D(2, 2)(x)
    x = layers.GlobalAvgPool1D(name='pooled')(x)
    x = layers.Dropout(drop)(x)
    out = layers.Dense(1, activation='sigmoid', kernel_regularizer=reg, name='det')(x)
    return keras.Model(inp, out)

def build_recording_model(window_model):
    """Official-equivalent stage 2: frozen pooled backbone, new dropout + sigmoid dense layer."""
    backbone = keras.Model(window_model.input, window_model.get_layer('pooled').output)
    backbone.trainable = False
    inp = layers.Input((None, 18))
    x = backbone(inp)
    x = layers.Dropout(0.25)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inp, out)

def official_windows(indices, feats):
    """Exact endpoint convention from Original.ipynb (exact-ending window excluded)."""
    X, y = [], []
    for i in indices:
        s = feats[i]
        for st in range(0, len(s)-WIN, STRIDE):
            X.append(s[st:st+WIN]); y.append(records[i]['label'])
    return np.asarray(X, np.float32), np.asarray(y, np.float32)

def pad_wrap(indices, feats):
    L = max(len(feats[i]) for i in indices)
    return np.stack([np.pad(feats[i], ((0, L-len(feats[i])), (0,0)), mode='wrap')
                     for i in indices]).astype(np.float32)

def callbacks():
    return [keras.callbacks.EarlyStopping('val_loss', patience=PATIENCE,
                                          restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.25,
                                              patience=RLR_PATIENCE, min_lr=MIN_LR)]

def fit_two_stage(builder, cfg, tr, va, feats, max_win=MAX_EPOCHS_WIN,
                  max_rec=MAX_EPOCHS_REC, verbose=0):
    """Official optimization, with caller-specified participant-disjoint tr/va."""
    model = builder(**cfg) if cfg else builder()
    model.compile(optimizer=keras.optimizers.Adam(LR_WIN, clipvalue=0.5),
                  loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=['accuracy'])
    Xtr, ytr = official_windows(tr, feats)
    h1 = model.fit(Xtr, ytr, batch_size=BATCH_WIN, epochs=max_win,
                   validation_data=(pad_wrap(va, feats), labels(va)),
                   callbacks=callbacks(), verbose=verbose)
    full = build_recording_model(model)
    full.compile(optimizer=keras.optimizers.Adam(LR_REC, clipvalue=1),
                 loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                 metrics=['accuracy'])
    h2 = full.fit(pad_wrap(tr, feats), labels(tr), batch_size=BATCH_REC, epochs=max_rec,
                  validation_data=(pad_wrap(va, feats), labels(va)),
                  callbacks=callbacks(), verbose=verbose)
    best = dict(window=int(np.argmin(h1.history['val_loss'])+1),
                recording=int(np.argmin(h2.history['val_loss'])+1))
    return model, full, h1.history, h2.history, best

def refit_two_stage(builder, cfg, train_idx, feats, epochs, verbose=0):
    """After epoch selection, refit on every outer-training participant without test data."""
    model = builder(**cfg) if cfg else builder()
    model.compile(optimizer=keras.optimizers.Adam(LR_WIN, clipvalue=0.5),
                  loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=['accuracy'])
    X, y = official_windows(train_idx, feats)
    model.fit(X, y, batch_size=BATCH_WIN, epochs=max(1, epochs['window']), verbose=verbose)
    full = build_recording_model(model)
    full.compile(optimizer=keras.optimizers.Adam(LR_REC, clipvalue=1),
                 loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                 metrics=['accuracy'])
    full.fit(pad_wrap(train_idx, feats), labels(train_idx), batch_size=BATCH_REC,
             epochs=max(1, epochs['recording']), verbose=verbose)
    return model, full

def predict_recordings(full, indices, feats):
    return np.array([float(full.predict(feats[i][None], verbose=0)[0,0]) for i in indices])

def predict_window_average(model, indices, feats):
    out = []
    for i in indices:
        X, _ = official_windows([i], feats)
        out.append(float(model.predict(X, verbose=0).mean()))
    return np.array(out)

_m = build_lpgnet()
print(f'LPGNet: {_m.count_params():,} total parameters; '
      f'{sum(int(w.numpy().size) for w in _m.trainable_weights):,} trainable; '
      f'{_m.count_params()+18*LP_ORDER:,} including LP coefficients')
del _m


In [ ]:
# ============================================================
# OOF metrics, participant aggregation, errors, and clinical subgroups
# ============================================================
def metric_dict(y, p):
    y = np.asarray(y, int); p = np.asarray(p, float); yh = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    return dict(n=len(y), accuracy=accuracy_score(y, yh),
                balanced_accuracy=balanced_accuracy_score(y, yh),
                macro_f1=f1_score(y, yh, average='macro', zero_division=0),
                f1_pd=f1_score(y, yh, zero_division=0),
                sensitivity=tp/(tp+fn) if tp+fn else np.nan,
                specificity=tn/(tn+fp) if tn+fp else np.nan,
                roc_auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
                log_loss=log_loss(y, np.clip(p,1e-7,1-1e-7), labels=[0,1]))

def oof_frame(rows):
    df = pd.DataFrame(rows)
    df['prediction'] = (df['probability'] >= 0.5).astype(int)
    df['correct'] = (df['prediction'] == df['label']).astype(int)
    df['confidence'] = np.maximum(df['probability'], 1-df['probability'])
    df['nll'] = -(df['label']*np.log(np.clip(df['probability'],1e-7,1)) +
                  (1-df['label'])*np.log(np.clip(1-df['probability'],1e-7,1)))
    for col in ('study','diagnosis','hy','hy_label','hy_status','age','gender'):
        df[col] = df['subject'].map(lambda s: clinical[s][col])
    return df

def participant_oof(df, group_cols):
    """Equal weight per participant; probability is averaged over that person's walks."""
    keys = group_cols + ['subject']
    agg = dict(label=('label','first'), fold=('fold','first'), probability=('probability','mean'),
               recordings=('file','nunique'))
    out = df.groupby(keys, as_index=False).agg(**agg)
    return oof_frame(out.to_dict('records'))

def summarize_oof(df, group_cols):
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        rec = metric_dict(g['label'], g['probability'])
        part = participant_oof(g, group_cols)
        pm = metric_dict(part['label'], part['probability'])
        rows.append({**base, **{'recording_'+k:v for k,v in rec.items()}})
        rows[-1].update({'participant_'+k:v for k,v in pm.items()})
    return pd.DataFrame(rows)

def wilson_interval(correct, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = correct/n; den = 1+z*z/n
    center = (ph+z*z/(2*n))/den
    half = z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    return center-half, center+half

def error_tables(df, group_cols):
    part = participant_oof(df, group_cols)
    errors = part[part['correct'] == 0].sort_values(group_cols+['nll'], ascending=[True]*len(group_cols)+[False])
    pd_rows = []
    for keys, g in part[part['label'] == 1].groupby(group_cols+['hy_label'], dropna=False):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['hy_label'], keys))
        n = len(g); correct = int(g['correct'].sum()); lo, hi = wilson_interval(correct, n)
        pd_rows.append({**base, 'pd_participants':n, 'correct':correct, 'errors':n-correct,
                        'pd_sensitivity':correct/n, 'wilson_95_low':lo, 'wilson_95_high':hi})
    study_rows = []
    for keys, g in part.groupby(group_cols+['study','diagnosis']):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['study','diagnosis'], keys))
        study_rows.append({**base, **metric_dict(g['label'], g['probability'])})
    return part, errors, pd.DataFrame(pd_rows), pd.DataFrame(study_rows)

def show_table(df, title, limit=100):
    print('\n'+title)
    display(df.head(limit))

def pooled_bootstrap(df, group_cols, metrics=('accuracy','balanced_accuracy','macro_f1','sensitivity','specificity','roc_auc')):
    """Stratified participant bootstrap conditional on fitted OOF predictions."""
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        part = participant_oof(g, group_cols).reset_index(drop=True)
        y = part['label'].to_numpy(); p = part['probability'].to_numpy()
        class_idx = [np.flatnonzero(y == c) for c in (0,1)]
        rng = np.random.default_rng(2026)
        for metric in metrics:
            observed = metric_dict(y,p)[metric]
            vals = []
            for _ in range(N_BOOT):
                ii = np.concatenate([rng.choice(ix, len(ix), replace=True) for ix in class_idx])
                vals.append(metric_dict(y[ii],p[ii])[metric])
            rows.append({**base,'metric':metric,'estimate':observed,
                         'bootstrap_95_low':np.nanpercentile(vals,2.5),
                         'bootstrap_95_high':np.nanpercentile(vals,97.5)})
    return pd.DataFrame(rows)

def plot_confusion_and_roc(df, group_cols, title_col):
    part = participant_oof(df, group_cols)
    groups = list(part.groupby(group_cols))
    fig, axes = plt.subplots(len(groups), 2, figsize=(11,4.3*len(groups)), squeeze=False)
    for row, (keys,g) in enumerate(groups):
        if not isinstance(keys, tuple): keys=(keys,)
        title = ' | '.join(f'{c}={v}' for c,v in zip(group_cols,keys))
        cm = confusion_matrix(g['label'],g['prediction'],labels=[0,1])
        ConfusionMatrixDisplay(cm,display_labels=['Control','PD']).plot(ax=axes[row,0],colorbar=False)
        axes[row,0].set_title(title+' | participant OOF')
        fpr,tpr,_ = roc_curve(g['label'],g['probability'])
        axes[row,1].plot(fpr,tpr,label=f"AUC={roc_auc_score(g['label'],g['probability']):.3f}")
        axes[row,1].plot([0,1],[0,1],'--',color='gray')
        axes[row,1].set(xlabel='False-positive rate',ylabel='True-positive rate',title=title)
        axes[row,1].legend()
    fig.tight_layout(); plt.show()


In [ ]:
# ============================================================
# Fit global and fold-local feature extractors, then run the 2 x 2 study
# ============================================================
t0 = time.time()
global_C, global_stds = fit_lp([r['proc'] for r in records if r['label'] == 0])
global_feats = lp_features(range(len(records)), global_C, global_stds)
local_feats = []
for k, f in enumerate(FOLDS):
    C, stds = fit_lp([records[i]['proc'] for i in f['train'] if records[i]['label'] == 0])
    local_feats.append(lp_features(f['train']+f['test'], C, stds))
print(f'Feature preparation: {(time.time()-t0)/60:.1f} min')

conditions = [('global','test'), ('local','test'), ('global','inner'), ('local','inner')]
oof_rows, fold_rows, run_histories = [], [], {}
for lp_scope, validation in conditions:
    condition = f'{lp_scope}__{validation}'
    tf.keras.utils.set_random_seed(SEED)  # same starting random stream for every condition
    for k, f in enumerate(FOLDS):
        keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED+k)
        start = time.time()
        feats = global_feats if lp_scope == 'global' else local_feats[k]
        if validation == 'test':
            tr, va = f['train'], f['test']
        else:
            tr, va = split_val(f['train'], seed=SEED+k)
        model, full, h1, h2, best = fit_two_stage(build_lpgnet, {}, tr, va, feats)
        p = predict_recordings(full, f['test'], feats)
        for i, prob in zip(f['test'], p):
            oof_rows.append(dict(condition=condition, lp_scope=lp_scope, validation=validation,
                                 fold=k, file=records[i]['file'], subject=records[i]['subject'],
                                 label=records[i]['label'], probability=prob))
        m = metric_dict(labels(f['test']), p)
        fold_rows.append(dict(condition=condition, fold=k, best_window_epoch=best['window'],
                              best_recording_epoch=best['recording'], **m))
        run_histories[(condition,k)] = (h1,h2)
        print(f"{condition:15s} fold {k}: rec-AUC={m['roc_auc']:.3f} "
              f"ACC={m['accuracy']:.3f} F1-PD={m['f1_pd']:.3f} "
              f"SENS={m['sensitivity']:.3f} SPEC={m['specificity']:.3f} "
              f"({time.time()-start:.0f}s)")

oof = oof_frame(oof_rows)
fold_metrics = pd.DataFrame(fold_rows)
print(f'Total training time: {(time.time()-t0)/60:.1f} min')
show_table(fold_metrics.round(4), 'Fold-level recording metrics (descriptive; fold SD is not a CI)')


In [ ]:
# ============================================================
# Complete OOF summaries: recordings for official comparison, participants primary
# ============================================================
summary = summarize_oof(oof, ['condition'])
show_table(summary.round(4), 'Pooled OOF metrics — every participant tested once per condition')
participant_ci = pooled_bootstrap(oof, ['condition'])
show_table(participant_ci.round(4), 'Participant OOF metrics with 95% stratified bootstrap intervals')

fold_dispersion = (fold_metrics.groupby('condition')[['roc_auc','accuracy','f1_pd','sensitivity','specificity']]
                   .agg(['mean','std']).round(4))
show_table(fold_dispersion, 'Fold mean ± sample SD (not confidence intervals)')

paper = pd.DataFrame({'paper_reported_mean':[0.917,0.903,0.932],
                      'paper_reported_sd':[0.094,0.058,0.036]},
                     index=['roc_auc','accuracy','f1_pd'])
official_stats=fold_metrics[fold_metrics.condition=='global__test'][['roc_auc','accuracy','f1_pd']].agg(['mean','std']).T
paper['reconstructed_mean']=official_stats['mean']
paper['reconstructed_sd']=official_stats['std']
show_table(paper.round(4), 'Published values and global__test reconstruction; not expected to be bit-identical')


In [ ]:
# ============================================================
# Paired participant-level leakage contrasts
# ============================================================
contrasts = [
    ('global__test','local__test','Global minus fold-local LP | test validation'),
    ('global__inner','local__inner','Global minus fold-local LP | training-only validation'),
    ('global__test','global__inner','Test minus training-only validation | global LP'),
    ('local__test','local__inner','Test minus training-only validation | fold-local LP'),
    ('global__test','local__inner','Released minus corrected | combined difference')]

def paired_condition_bootstrap(df, contrasts, metrics=('accuracy','balanced_accuracy','f1_pd','sensitivity','specificity','roc_auc')):
    part = participant_oof(df, ['condition'])
    rows = []
    for a,b,label in contrasts:
        wide = part[part.condition.isin([a,b])].pivot(index='subject',columns='condition',values='probability')
        meta = part[part.condition == a].set_index('subject').loc[wide.index]
        y = meta['label'].to_numpy(); pa=wide[a].to_numpy(); pb=wide[b].to_numpy()
        classes = [np.flatnonzero(y == c) for c in (0,1)]
        rng = np.random.default_rng(2027)
        for metric in metrics:
            observed = metric_dict(y,pa)[metric]-metric_dict(y,pb)[metric]
            vals=[]
            for _ in range(N_BOOT):
                ii=np.concatenate([rng.choice(ix,len(ix),replace=True) for ix in classes])
                vals.append(metric_dict(y[ii],pa[ii])[metric]-metric_dict(y[ii],pb[ii])[metric])
            rows.append(dict(contrast=label,metric=metric,difference=observed,
                             bootstrap_95_low=np.nanpercentile(vals,2.5),
                             bootstrap_95_high=np.nanpercentile(vals,97.5)))
    return pd.DataFrame(rows)

leakage_contrasts = paired_condition_bootstrap(oof, contrasts)
show_table(leakage_contrasts.round(4),
           'Paired differences (first minus second); positive accuracy/sensitivity/specificity favors first')

# Fold-paired deltas are descriptive and do not isolate stochastic training variability.
for a,b,label in contrasts:
    aa=fold_metrics[fold_metrics.condition==a].set_index('fold')
    bb=fold_metrics[fold_metrics.condition==b].set_index('fold')
    d=aa[['roc_auc','accuracy','f1_pd','sensitivity','specificity']]-bb[['roc_auc','accuracy','f1_pd','sensitivity','specificity']]
    print('\n'+label)
    display(pd.DataFrame({'mean_delta':d.mean(),'fold_sd':d.std()}).round(4))


In [ ]:
# ============================================================
# Confusion matrices, misclassified participants, and HY patterns
# ============================================================
plot_confusion_and_roc(oof, ['condition'], 'condition')
participants, participant_errors, pd_severity, by_study = error_tables(oof, ['condition'])
show_table(participant_errors[['condition','fold','subject','study','diagnosis','hy_label','age',
                               'recordings','probability','prediction','confidence','nll']].round(4),
           'Misclassified outer-test participants (all conditions)', limit=200)
show_table(pd_severity.round(4),
           'PD sensitivity by verified Hoehn–Yahr stage; Unknown retained, controls not assigned HY=0')
show_table(by_study.round(4), 'Participant OOF results by contributing study and diagnosis')

error_frequency = (participant_errors.groupby('subject').agg(
    error_conditions=('condition','nunique'), conditions=('condition',lambda x:', '.join(x)),
    mean_nll=('nll','mean')).reset_index())
error_frequency = error_frequency.merge(participants[['subject','study','diagnosis','hy_label','age']].drop_duplicates('subject'),
                                        on='subject',how='left')
show_table(error_frequency.sort_values(['error_conditions','mean_nll'],ascending=[False,False]).round(4),
           'Participants misclassified across leakage conditions')

# Every participant remains inspectable, including correctly classified participants.
show_table(participants[['condition','fold','subject','diagnosis','hy_label','probability','prediction',
                         'correct','recordings','nll']].sort_values(['condition','nll'],ascending=[True,False]).round(4),
           'Complete participant OOF table ranked by individual NLL within condition', limit=40)


In [ ]:
# ============================================================
# Export exact OOF predictions and diagnostics for auditing
# ============================================================
outdir = Path('lpgnet_leakage_outputs'); outdir.mkdir(exist_ok=True)
for name, df in [('recording_oof',oof),('participant_oof',participants),
                 ('fold_metrics',fold_metrics),('pooled_metrics',summary),
                 ('participant_bootstrap',participant_ci),('leakage_contrasts',leakage_contrasts),
                 ('participant_errors',participant_errors),('pd_severity',pd_severity),
                 ('study_metrics',by_study),('error_frequency',error_frequency)]:
    df.to_csv(outdir/f'{name}.csv',index=False)
print('CSV outputs written to:',outdir.resolve())


## Interpretation and reporting limits

- `global__test` follows the released computational protocol, but numerical reproduction can
  still vary with TensorFlow/Keras version, GPU nondeterminism, and random-number consumption.
- Global LP fitting uses control gait from outer-test participants. It is unsupervised with
  respect to PD labels but not leakage-free. Its effect on sensitivity or specificity is an
  empirical result, not a guaranteed one-sided direction.
- Test-fold early stopping uses outer-test labels for checkpoint and learning-rate decisions.
- `local__inner` protects outer-test participants from both mechanisms. Its early-stopping
  participants still contribute to the outer-training LP fit; this does not touch outer tests,
  but stricter inner-validation experiments can fit LP only on the window-training subset.
- Recording metrics reproduce the official estimand. Participant metrics give every person
  equal weight and are primary for diagnostic generalization.
- Bootstrap intervals resample participants conditional on fitted OOF predictions. They do
  not include refitting, new-split, acquisition-site, or external-population uncertainty.
- HY subgroup analyses are exploratory PD-sensitivity descriptions with small cells and
  pointwise Wilson intervals, not severity-classification accuracy or confirmatory tests.
